# Canonicalize our gold segments (Gemma-4-31B, Tomer's prompt)

Runs the NLP_ADV canonicalization over **our** 523 Step-7 gold segments so we can
compare Step-10 linking on RAW vs CANONICAL text of the same segments.

**Before running:** upload `canon_input.json` (523 segments, exported on the cluster by
`steps/12_joint_pipeline/code/export_for_canon.py`) to your Drive at the `IN_PATH` below,
and set an `HF_TOKEN` Colab secret. Runtime: A100, 4-bit. This is a long job (~hours);
it is **resumable** -- re-run the batch cell after any disconnect and it skips finished
segments. Output `canonical_gold.json` is rewritten to Drive after every segment.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# === config ===
from pathlib import Path
from google.colab import userdata
BASE     = Path('/content/drive/MyDrive/NLP ADVANCED/FinalProject')
IN_PATH  = BASE / 'canon_input.json'          # <- upload our export here
OUT_PATH = BASE / 'canonical_gold.json'        # <- result (pull back to cluster)
CFG = dict(model_id='google/gemma-4-31B-it', quant_4bit=True, max_new=1024, cap_words=3500)
assert IN_PATH.exists(), f'upload canon_input.json to {IN_PATH}'
print(CFG)

In [ ]:
# === load Gemma-4-31B-it (4-bit nf4; fits ~18-20GB on an A100) ===
import torch
from huggingface_hub import login
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig
login(token=userdata.get('HF_TOKEN'))
processor = AutoProcessor.from_pretrained(CFG['model_id'])
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.bfloat16)
model_hf = AutoModelForMultimodalLM.from_pretrained(
    CFG['model_id'], quantization_config=bnb, device_map='auto')
model_hf.eval()
print('loaded | GPU mem', round(torch.cuda.memory_allocated()/1e9,1),'GB')

In [ ]:
# === Hebrew-robust JSON extraction (Tomer's helpers) ===
import re
def _json_objects(s):
    depth,start,in_str,esc = 0,None,False,False
    for i,ch in enumerate(s):
        if in_str:
            if esc: esc=False
            elif ch=='\\\\': esc=True
            elif ch=='"': in_str=False
            continue
        if ch=='"': in_str=True
        elif ch=='{':
            if depth==0: start=i
            depth+=1
        elif ch=='}':
            if depth>0:
                depth-=1
                if depth==0 and start is not None: yield s[start:i+1]
def _salvage_json(s):
    return re.sub(r'(?<=[\u0590-\u05FF])"(?=[\u0590-\u05FF])', '\\\\"', s)

In [ ]:
# === canonicalization prompt (Tomer's exact CANON_SYSTEM + CANON_INSTRUCTIONS) ===
import json
CANON_SYSTEM = ('אתה עורך לשוני. אתה מקבל מקטע דיון מוועדת הכספים של הכנסת, ובו דוברים שונים '
                'בסגנונות ואוצר מילים שונים. תפקידך לנסח מחדש את תוכן הדיון בלשון אחידה, '
                'ניטרלית ועניינית - לשמר את כל המידע (נושאים, עמדות, נתונים, החלטות) אך '
                'להסיר סגנון אישי, רטוריקה, ומאפייני דובר. אתה לא מסכם ולא מקצר - אתה מנסח מחדש.')
CANON_INSTRUCTIONS = '''נסח מחדש את מקטע הדיון הבא בלשון אחידה וניטרלית.

עקרונות:
- שמר את כל התוכן העובדתי: מה נדון, אילו עמדות הוצגו, נתונים מספריים, החלטות והצבעות.
- אחֵד את הרגיסטר: אותו אוצר מילים ענייני לכל הדוברים, ללא סלנג, רטוריקה, ברכות או ציטוט סגנוני.
- אל תסכם ואל תקצר באופן אגרסיבי - שמור על אורך דומה לתוכן המהותי של המקטע.
- כתוב כטקסט רציף בגוף שלישי ("הוצגה עמדה ש...", "סוכם כי...", "התקיימה הצבעה ש...").
- אל תוסיף פרשנות או מידע שאינו במקטע.

החזר JSON יחיד בלבד:
{"subject": "<כותרת נושא קצרה 3-6 מילים>", "canonical": "<הניסוח המחדש הניטרלי>", "decision": "<אושר/נדחה/ללא הצבעה/נדחה להמשך>", "amounts": ["<סכומים או מספרי פניות>"]}'''

@torch.no_grad()
def _generate_canon(prompt):
    messages=[{'role':'system','content':[{'type':'text','text':CANON_SYSTEM}]},
              {'role':'user','content':[{'type':'text','text':prompt}]}]
    inputs=processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
                                         return_dict=True, return_tensors='pt').to(model_hf.device)
    out=model_hf.generate(**inputs, max_new_tokens=CFG['max_new'], do_sample=False)
    return processor.decode(out[0, inputs['input_ids'].shape[-1]:], skip_special_tokens=True)

def _first_canon_json(raw):
    for obj in _json_objects(raw):
        for cand in (obj, _salvage_json(obj)):
            try: d=json.loads(cand)
            except json.JSONDecodeError: continue
            if 'canonical' in d: return d
    raise ValueError('no canonical json')

def canonicalize(raw_txt):
    raw=_generate_canon(f"{CANON_INSTRUCTIONS}\n\n--- המקטע ---\n{raw_txt}")
    try: c=_first_canon_json(raw)
    except Exception: c={'subject':'','canonical':raw.strip(),'decision':'','amounts':[]}
    return c

In [ ]:
# === resumable batch: canonicalize every segment, rewrite output after each ===
seg_in = json.load(open(IN_PATH, encoding='utf-8'))
done = {c['seg_key']: c for c in (json.load(open(OUT_PATH,encoding='utf-8')) if OUT_PATH.exists() else [])}
print(f'{len(done)}/{len(seg_in)} already done; resuming')
for n,r in enumerate(seg_in):
    if r['seg_key'] in done: continue
    raw_txt=' '.join(r['raw'].split()[:CFG['cap_words']])
    c=canonicalize(raw_txt)
    done[r['seg_key']]={'seg_key':r['seg_key'],'date':r['date'],
                        'subject':c.get('subject',''),'canonical':c.get('canonical',''),
                        'decision':c.get('decision',''),'amounts':c.get('amounts',[])}
    json.dump(list(done.values()), open(OUT_PATH,'w',encoding='utf-8'), ensure_ascii=False)
    if (len(done))%10==0: print(f'  {len(done)}/{len(seg_in)}')
print('DONE:', len(done), 'segments ->', OUT_PATH)

## Get the result back to the cluster

**Option A (simplest):** download `canonical_gold.json` from Drive and place it at
`ANLP-PROJECT/steps/12_joint_pipeline/outputs/canonical_gold.json` on the cluster.

**Option B (direct scp):** run the cell below with your cluster SSH set up in Colab
(add your private key + known_hosts). This pushes straight to the cluster, no manual step.

In [ ]:
# Option B: scp the result directly to the cluster (edit USER/HOST/PATH; needs an SSH key in Colab)
# !pip -q install scp paramiko
# USER, HOST = 'yoel.marcu2003', 'river.cs.huji.ac.il'   # or your CS SSH gateway
# DEST = '/cs/labs/daphna/yoel.marcu2003/ANLP-PROJECT/steps/12_joint_pipeline/outputs/canonical_gold.json'
# !scp -o StrictHostKeyChecking=no '{OUT_PATH}' {USER}@{HOST}:{DEST}
print('see comments to enable direct scp')